In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.sql("select * from bronze.grocery_raw_new")
df.display()

In [0]:
from pyspark.sql.functions import col

table_name = "silver.grocery_raw_new"
if spark.catalog.tableExists(table_name):
    latest_date = spark.sql(f"SELECT MAX(Inserted_date) as max_new_date FROM {table_name}").collect()[0]['max_new_date']
    latest_records_df = df.filter(col('Inserted_date') == latest_date)
else:
    latest_records_df = df

display(latest_records_df)

In [0]:
silver_df = latest_records_df.withColumn("reorder_category",when(col("Reorder_Quantity")>=100,"High").when(col("Reorder_Quantity")>=50,"Medium").otherwise("Low"))
display(silver_df)

In [0]:
silver_df= silver_df.dropDuplicates()

In [0]:
silver_df.write.format('delta').mode('overwrite').saveAsTable("silver.grocery_silver_new")

In [0]:
%sql
select * from silver.grocery_silver_new